In [41]:
# Importando as bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações para melhor visualização
plt.style.use('default')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
np.set_printoptions(suppress=True)

In [42]:
# Lendo o arquivo CSV em um DataFrame
df = pd.read_csv('movies.csv', sep='|')
df.head()

C:\Users\Rhamon-Temp\AppData\Local\Temp\ipykernel_6180\2025610993.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('movies.csv', sep='|')


,id,tituloPincipal,tituloOriginal,anoLancamento,tempoMinutos,genero,notaMedia,numeroVotos,generoArtista,personagem,nomeArtista,anoNascimento,anoFalecimento,profissao,titulosMaisConhecidos
0,tt0000009,Miss Jerry,Miss Jerry,1894,45,Romance,5.30,200,actress,Miss Geraldine Holbrook (Miss Jerry),Blanche Bayliss,1878,1951,actress,tt0000009
1,tt0000009,Miss Jerry,Miss Jerry,1894,45,Romance,5.30,200,actor,Mr. Hamilton,William Courtenay,1875,1933,actor,"tt0021535,tt0000009,tt0020355,tt0020403"
2,tt0000009,Miss Jerry,Miss Jerry,1894,45,Romance,5.30,200,actor,Chauncey Depew - the Director of the New York ...,Chauncey Depew,1834,1928,"actor,writer","tt1076833,tt0490842,tt0000009,tt4484306"
3,tt0000502,Bohemios,Bohemios,1905,100,\N,4.20,14,actor,\N,Antonio del Pozo,\N,\N,actor,tt0000502
4,tt0000502,Bohemios,Bohemios,1905,100,\N,4.20,14,actor,\N,El Mochuelo,\N,\N,actor,tt0000502


In [43]:
# Definindo os tipos corretos para cada coluna
df = df.replace('\\N', np.nan)

# Convertendo colunas numéricas
colunas_numericas = ['anoLancamento', 'tempoMinutos', 'anoNascimento', 'anoFalecimento']
for coluna in colunas_numericas:
    df[coluna] = pd.to_numeric(df[coluna], errors='coerce').astype('Int64')

# Convertendo colunas de texto
colunas_texto = ['id', 'tituloPincipal', 'tituloOriginal', 'genero', 'generoArtista',
                'personagem', 'nomeArtista', 'profissao', 'titulosMaisConhecidos']
for coluna in colunas_texto:
    df[coluna] = df[coluna].astype('string')

df.dtypes

id                       string[python]
tituloPincipal           string[python]
tituloOriginal           string[python]
anoLancamento                     Int64
tempoMinutos                      Int64
genero                   string[python]
notaMedia                       float64
numeroVotos                       int64
generoArtista            string[python]
personagem               string[python]
nomeArtista              string[python]
anoNascimento                     Int64
anoFalecimento                    Int64
profissao                string[python]
titulosMaisConhecidos    string[python]
dtype: object

In [44]:
# Processamento dos dados
# 1. Eliminar IDs duplicados
df_unicos = df.drop_duplicates(subset=['id'], keep='first').copy()

# 2. Padronizar gêneros (usando .loc para evitar warning)
df_unicos.loc[:, 'genero_padronizado'] = df_unicos['genero'].str.lower().str.strip()

# 3. Filtrar apenas Romance e/ou Drama
df_romance_drama = df_unicos[
    df_unicos['genero_padronizado'].str.contains('romance|drama', case=False, na=False)
].copy()

print(f"Quantidade de filmes com Romance e/ou Drama: {len(df_romance_drama):,}")
df_romance_drama.head()

Quantidade de filmes com Romance e/ou Drama: 135,727


,id,tituloPincipal,tituloOriginal,anoLancamento,tempoMinutos,genero,notaMedia,numeroVotos,generoArtista,personagem,nomeArtista,anoNascimento,anoFalecimento,profissao,titulosMaisConhecidos,genero_padronizado
0,tt0000009,Miss Jerry,Miss Jerry,1894,45,Romance,5.30,200,actress,Miss Geraldine Holbrook (Miss Jerry),Blanche Bayliss,1878,1951,actress,tt0000009,romance
9,tt0000591,The Prodigal Son,L'enfant prodigue,1907,90,Drama,5.10,20,actor,<NA>,Henri Gouget,<NA>,<NA>,actor,"tt3752844,tt0451296,tt0311681,tt0400955",drama
13,tt0000615,Robbery Under Arms,Robbery Under Arms,1907,<NA>,Drama,4.30,23,actress,<NA>,Rhoda Dendron,1884,1970,actress,tt0000615,drama
19,tt0000630,Hamlet,Amleto,1908,<NA>,Drama,2.90,25,actress,Ofelia,Fernanda Negri Pouget,1889,1955,actress,"tt0003489,tt0184691,tt0167871,tt0000630",drama
30,tt0000886,"Hamlet, Prince of Denmark",Hamlet,1910,<NA>,Drama,4.70,38,actor,Hamlet,Jean Mounet-Sully,1841,1916,actor,"tt0449855,tt0000886,tt1732180,tt0492275",drama


In [ ]:
# Aplicando filtro baseado no 1% mais votado

""" 1. Identificar o 1% dos filmes mais votados (99º percentil) """
limite_99_percentil = df_romance_drama['numeroVotos'].quantile(0.99)

""" 2. Definir o limite como 1/10 do limite do 99º percentil """
limite_final = limite_99_percentil / 10

""" 3. Aplicar o filtro final """
df_filtrado = df_romance_drama[df_romance_drama['numeroVotos'] >= limite_final].copy()

print(f"Quantidade de filmes filtrados: {len(df_filtrado):,}")
df_filtrado.head()

Quantidade de filmes filtrados: 7,117


,id,tituloPincipal,tituloOriginal,anoLancamento,tempoMinutos,genero,notaMedia,numeroVotos,generoArtista,personagem,nomeArtista,anoNascimento,anoFalecimento,profissao,titulosMaisConhecidos,genero_padronizado
1229,tt0004972,The Birth of a Nation,The Birth of a Nation,1915,195,"Drama,War",6.20,24998,actress,Elsie - Stoneman's Daughter,Lillian Gish,1893,1993,"actress,writer,soundtrack","tt0038499,tt0019585,tt0094315,tt0048424","drama,war"
2434,tt0006864,Intolerance,Intolerance: Love's Struggle Throughout the Ages,1916,163,"Drama,History",7.70,15749,actress,The Woman Who Rocks the CradleEternal Mother,Lillian Gish,1893,1993,"actress,writer,soundtrack","tt0038499,tt0019585,tt0094315,tt0048424","drama,history"
4788,tt0009968,Broken Blossoms,Broken Blossoms or The Yellow Man and the Girl,1919,90,"Drama,Romance",7.30,10479,actor,Battling Burrows,Donald Crisp,1882,1974,"actor,director,writer","tt0009968,tt0030287,tt0032145,tt0033729","drama,romance"
7329,tt0012349,The Kid,The Kid,1921,68,"Comedy,Drama,Family",8.30,127040,actress,The Woman,Edna Purviance,1895,1958,actress,"tt0012349,tt0014624,tt0008133,tt0017575","comedy,drama,family"
7359,tt0012364,The Phantom Carriage,Körkarlen,1921,107,"Drama,Fantasy,Horror",8.00,12817,actor,Georges,Tore Svennberg,1858,1941,actor,"tt0030337,tt0011370,tt0033111,tt0012364","drama,fantasy,horror"


In [48]:
# Removendo colunas desnecessárias e exportando para CSV
colunas_remover = [
    'tituloOriginal', 'generoArtista', 'personagem', 'nomeArtista', 'anoLancamento',
    'anoNascimento', 'anoFalecimento', 'profissao', 'titulosMaisConhecidos',
    'genero_padronizado'
]

df_final = df_filtrado.drop(columns=colunas_remover)

# Exportando para CSV
df_final.to_csv('filmes_romance_drama_filtrados.csv', index=False)

print(f"Arquivo exportado: filmes_romance_drama_filtrados.csv")
print(f"Colunas finais: {list(df_final.columns)}")
df_final.head()

Arquivo exportado: filmes_romance_drama_filtrados.csv
Colunas finais: ['id', 'tituloPincipal', 'tempoMinutos', 'genero', 'notaMedia', 'numeroVotos']


,id,tituloPincipal,tempoMinutos,genero,notaMedia,numeroVotos
1229,tt0004972,The Birth of a Nation,195,"Drama,War",6.20,24998
2434,tt0006864,Intolerance,163,"Drama,History",7.70,15749
4788,tt0009968,Broken Blossoms,90,"Drama,Romance",7.30,10479
7329,tt0012349,The Kid,68,"Comedy,Drama,Family",8.30,127040
7359,tt0012364,The Phantom Carriage,107,"Drama,Fantasy,Horror",8.00,12817
